In [1]:
!ls

01_study_ds.ipynb	     03_split_ds_yolo.ipynb  docs
02_data_visualization.ipynb  04_train_models.ipynb   videp.mp4


In [2]:
train_json_path = '../dataset/processed/experiments/set1_balanced_subsampled/fold_1_train_coco.json'
val_json_path = '../dataset/processed/experiments/set1_balanced_subsampled/fold_1_val_coco.json'
test_json_path = '../dataset/processed/experiments/set1_balanced_subsampled/test_coco.json'

In [3]:
from dataset.efficientdet_dataset import CocoDataset
from torch.utils.data import DataLoader
from utils.data_augmentations import get_train_transforms, get_valid_transforms

/root/MURIA/Object_Recognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_ds = CocoDataset(train_json_path, transform=get_train_transforms())
val_ds = CocoDataset(val_json_path, transform=get_valid_transforms())
test_ds = CocoDataset(test_json_path, transform=get_valid_transforms())


loading annotations into memory...
Done (t=0.10s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


/root/MURIA/Object_Recognition/.venv/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
from dataset.efficientdet_dataset import collate_fn
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4, collate_fn=collate_fn, pin_memory=True)

In [6]:
from models.efficientdet import EfficientDetModel
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Instanciar el modelo (3 clases: horse, penguin, pig)
my_detector = EfficientDetModel(model_name='efficientdet_d0', num_classes=3, image_size=(512, 512), bench_task='')
model_train = my_detector.get_train_model(device=device)

# 2. Optimizador (AdamW suele funcionar muy bien con EfficientDet)
optimizer = torch.optim.AdamW(model_train.parameters(), lr=2e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# 3. Scheduler (opcional, para bajar el LR si el entrenamiento se estanca)

In [8]:
def validate_one_epoch(model, loader, device):
    model.eval() # Modo evaluación
    running_loss = 0.0
    
    with torch.no_grad(): # No calculamos gradientes para ahorrar memoria
        for images, targets in loader:
            # Preparar tensores
            images = torch.stack(images).to(device).float()
            boxes = [t['bbox'].to(device) for t in targets]
            classes = [t['cls'].to(device) for t in targets]
            
            target_res = {
                'bbox': boxes,
                'cls': classes
            }
            
            # El Bench de entrenamiento nos devuelve la pérdida de validación
            output = model(images, target_res)
            loss = output['loss']
            running_loss += loss.item()
            
    return running_loss / len(loader)

In [9]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    
    # Envolvemos el loader con tqdm para la barra de progreso
    # 'desc' pone un texto a la izquierda, 'postfix' a la derecha
    pbar = tqdm(loader, total=len(loader), desc=f"Época {epoch+1}")
    
    for images, targets in pbar:
        # 1. Mover datos al dispositivo
        images = images.to(device).float()
        
        # 2. Preparar el padding de los targets (como vimos antes)
        max_boxes = max([t['bbox'].shape[0] for t in targets])
        batch_size = len(targets)
        
        batch_boxes = torch.zeros((batch_size, max_boxes, 4), device=device)
        batch_cls = torch.zeros((batch_size, max_boxes), device=device) - 1
        
        for i, t in enumerate(targets):
            n = t['bbox'].shape[0]
            if n > 0:
                batch_boxes[i, :n] = t['bbox']
                batch_cls[i, :n] = t['cls']

        target_res = {
            'bbox': batch_boxes,
            'cls': batch_cls
        }

        # 3. Optimización
        optimizer.zero_grad()
        loss_dict = model(images, target_res)
        loss = loss_dict['loss']
        
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        # 4. Actualizar la información de la barra en cada paso
        # Esto te permite ver el loss actual mientras entrena
        pbar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{total_loss/(pbar.n+1):.4f}")

    return total_loss / len(loader)

In [10]:
# src/models/evaluator.py (Puedes ponerlo en tu notebook 04)

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from models.efficientdet import get_predict_model

def evaluate_test_set(model_path, test_loader, device, num_classes=3):
    # 1. Cargamos el modelo en modo PREDICCIÓN
    # DetBenchPredict se encarga de aplicar NMS (Non-Maximum Suppression)
    model = get_predict_model(model_path, num_classes=num_classes)
    model.to(device)
    model.eval()
    
    metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
    
    with torch.no_grad():
        for images, targets in test_loader:
            images = torch.stack(images).to(device).float()
            
            # Predicción: devuelve [detenciones]
            # Cada detección es [x_min, y_min, x_max, y_max, score, class]
            outputs = model(images)
            
            preds = []
            for i in range(images.shape[0]):
                res = outputs[i]
                # Filtramos por un umbral de confianza (ej. 0.05)
                mask = res[:, 4] > 0.05
                preds.append({
                    'boxes': res[mask, :4],
                    'scores': res[mask, 4],
                    'labels': res[mask, 5].int()
                })
            
            # Formatear los targets reales para la métrica
            ground_truth = []
            for t in targets:
                # El dataset nos daba [y_min, x_min, y_max, x_max], 
                # torchmetrics suele preferir [x_min, y_min, x_max, y_max]
                gt_boxes = t['bbox'][:, [1, 0, 3, 2]] 
                ground_truth.append({
                    'boxes': gt_boxes.to(device),
                    'labels': t['cls'].to(device).int()
                })
            
            metric.update(preds, ground_truth)
    
    # Calcular resultado final
    result = metric.compute()
    return result

In [11]:
# Ejemplo rápido para visualizar en el notebook
def visualize_predictions(model, dataset, device, n_images=3):
    model.eval()
    for i in range(n_images):
        image, target = dataset[i]
        input_img = image.unsqueeze(0).to(device).float()
        
        with torch.no_grad():
            output = model(input_img)
        
        # Pintar cajas con tu script de visualization/visualize.py
        # ... (aquí llamarías a draw_annotations)

In [13]:
import torch
from tqdm.auto import tqdm # Para ver una barra de progreso bonita

# --- Configuración inicial ---
num_epochs = 6
best_val_loss = float('inf')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Historial para gráficas finales
history = {'train_loss': [], 'val_loss': []}

for epoch in range(num_epochs):
    # 1. FASE DE ENTRENAMIENTO
    model_train.train()
    train_running_loss = 0.0
    
    # tqdm nos ayuda a ver el progreso en tiempo real
    pbar = tqdm(train_loader, desc=f"Época {epoch+1}/{num_epochs} [TRAIN]")
    for images, targets in pbar:
        images = images.to(device).float()
        
        # Preparar targets para effdet
        target_res = {
            'bbox': [t['bbox'].to(device) for t in targets],
            'cls': [t['cls'].to(device) for t in targets]
        }
        
        optimizer.zero_grad()
        output = model_train(images, target_res)
        loss = output['loss']
        
        loss.backward()
        optimizer.step()
        
        train_running_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})

    avg_train_loss = train_running_loss / len(train_loader)
    scheduler.step() # Actualizar learning rate

    # 2. FASE DE VALIDACIÓN
    model_train.eval()
    val_running_loss = 0.0
    
    with torch.no_grad():
        vbar = tqdm(val_loader, desc=f"Época {epoch+1}/{num_epochs} [VAL]")
        for images, targets in vbar:
            images = torch.stack(images).to(device).float()
            target_res = {
                'bbox': [t['bbox'].to(device) for t in targets],
                'cls': [t['cls'].to(device) for t in targets]
            }
            
            output = model_train(images, target_res)
            val_running_loss += output['loss'].item()
            vbar.set_postfix({'val_loss': output['loss'].item()})

    avg_val_loss = val_running_loss / len(val_loader)
    
    # Guardar métricas
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    print(f"\nRESUMEN ÉPOCA {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # 3. LÓGICA DE GUARDADO (Checkpoints)
    
    # Guardar SIEMPRE el último modelo
    torch.save(model_train.model.state_dict(), 'last_model.pth')
    
    # Guardar el MEJOR modelo (si la loss de val ha bajado)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model_train.model.state_dict(), 'best_model.pth')
        print(f"⭐ ¡Nuevo mejor modelo guardado! (Val Loss: {best_val_loss:.4f})")

    # Guardar checkpoint cada 5 épocas por seguridad
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_train.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
        }, f'checkpoint_ep{epoch+1}.pth')

print("\n✅ Entrenamiento completado.")

Época 1/6 [TRAIN]:   0%|          | 0/232 [00:00<?, ?it/s]

Época 1/6 [VAL]:   0%|          | 0/129 [00:01<?, ?it/s]


TypeError: stack(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor

# Imports

In [1]:
from dataset.efficientdet_dataset import CocoDataset, collate_fn
from models.efficientdet import EfficientDetModel
from engine.train_efficientdet import train_efficientdet
from utils.data_augmentations import get_train_transforms, get_valid_transforms
from utils.file_utils import read_yaml
from torch.utils.data import DataLoader
import torch
import os
import re
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
CONFIG_PATH = PROJECT_ROOT / "paths.yaml"

/root/MURIA/Object_Recognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cargamos el Dataset

In [2]:
paths = read_yaml(CONFIG_PATH)
paths['dataset']['processed_path']

'./dataset/processed'

In [3]:
experiments_path = PROJECT_ROOT / paths['dataset']['processed_path'] / 'experiments'
experiments = sorted(os.listdir(experiments_path))

json_files = sorted([f for f in os.listdir(experiments_path / experiments[0]) if f.endswith('.json')])
json_files

['fold_1_train_coco.json',
 'fold_1_val_coco.json',
 'fold_2_train_coco.json',
 'fold_2_val_coco.json',
 'fold_3_train_coco.json',
 'fold_3_val_coco.json',
 'fold_4_train_coco.json',
 'fold_4_val_coco.json',
 'test_coco.json']

In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 2
BATCH_SIZE = 8
IMG_SIZE = (512, 512)
NUM_CLASSES = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-3
LR_SCHEDULER_STEP_SIZE = 10
LR_SCHEDULER_GAMMA = 0.1 

In [5]:
def get_fold_files(scenario_path):
    files = [f for f in os.listdir(scenario_path) if f.endswith('.json')]
    test_file = next((f for f in files if 'test' in f), None)
    folds = {}
    for f in files:
        match = re.search(r'fold_(\d+)_(\w+)_coco\.json', f)
        if match:
            fold_n = match.group(1)
            tipo = match.group(2) # 'train' o 'val'
            if fold_n not in folds: folds[fold_n] = {}
            folds[fold_n][tipo] = f
            
    return folds, test_file

get_fold_files(experiments_path / experiments[0])

({'1': {'val': 'fold_1_val_coco.json', 'train': 'fold_1_train_coco.json'},
  '2': {'val': 'fold_2_val_coco.json', 'train': 'fold_2_train_coco.json'},
  '3': {'train': 'fold_3_train_coco.json', 'val': 'fold_3_val_coco.json'},
  '4': {'val': 'fold_4_val_coco.json', 'train': 'fold_4_train_coco.json'}},
 'test_coco.json')

In [18]:
models_path = paths['models']['efficientdet']['path']


In [ ]:
from itertools import islice
models_path = paths['models']['efficientdet']['path']
for scenario_name in sorted(experiments[:1]):
    scenario_dir = experiments_path / scenario_name
    print(f"\n{'='*60}\n INICIANDO ESCENARIO: {scenario_name}\n{'='*60}")
    
    fold_dict, test_json = get_fold_files(scenario_dir)    

    for fold_idx in sorted(fold_dict.keys())[:1]:
        print(f"\n---Entrenando Fold {fold_idx} ---")

        folder_path = PROJECT_ROOT / models_path / scenario_name / f'Fold_{fold_idx}'
        print(folder_path)
        # Build paths
        train_path = scenario_dir / fold_dict[fold_idx]['train']
        val_path = scenario_dir / fold_dict[fold_idx]['val']
        test_path = scenario_dir / test_json    

        # Dataset y Loaders
        train_ds = CocoDataset(train_path, transform=get_train_transforms(IMG_SIZE))    
        val_ds = CocoDataset(val_path, transform=get_valid_transforms(IMG_SIZE))    
        test_ds = CocoDataset(test_path, transform=get_valid_transforms(IMG_SIZE))    

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

        # BORRAR LUEGO
        # train_loader = list(islice(train_loader, 2))
        # val_loader = list(islice(train_loader, 2))
        # test_loader = list(islice(train_loader, 2))

        detector = EfficientDetModel(model_name = 'efficientdet_d0', num_classes=NUM_CLASSES, image_size=IMG_SIZE)
        model_train = detector.get_train_model(device = DEVICE)

        optimizer = torch.optim.AdamW(model_train.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_SCHEDULER_STEP_SIZE, gamma=LR_SCHEDULER_GAMMA)
        summary = train_efficientdet(
            model=model_train,
            optimizer=optimizer,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            num_epochs=EPOCHS,
            num_classes=NUM_CLASSES,
            folder_path = folder_path,
            device=DEVICE,
        )
        print(summary)



 INICIANDO ESCENARIO: set1_balanced_subsampled

---Entrenando Fold 1 ---
/root/MURIA/Object_Recognition/models/efficientdet/set1_balanced_subsampled/Fold_1
